In [6]:
!pip install datasets
!pip install tiktoken

In [7]:
class SLMConfig:
    def __init__(self):
        self.vocab_size = 50257      # GPT-2 vocab
        self.n_blocks = 128          # Max sequence length (your model uses this)
        self.n_layers = 8            # Transformer blocks
        self.n_heads = 8             # Attention heads  
        self.d_emb = 512             # Embedding dim
        self.head_d_emb = 64         # 512/8 = 64 per head
        self.drop_rate = 0.1         # Dropout
        self.qkv_bias = True         # QKV bias
        self.is_debug = True   

In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math



# LayerNorm Inplementation (without bias) - as used in GPT-2
class LayerNorm(nn.Module):
    """Layer normalization applied across the embedding dimension."""
    
    def __init__(self, config: SLMConfig):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(config.d_emb))
        self.shift = nn.Parameter(torch.zeros(config.d_emb))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Normalize across embedding dimension
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        normalized_x = (x - mean) / (std + self.eps)
        return self.scale.to(x.device) * normalized_x + self.shift.to(x.device)


class GELU(nn.Module):
    """Gaussian Error Linear Unit activation."""

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)
        ))

# Feed Forward Network 

class FeedForward(nn.Module):
    """Position-wise feed-forward network."""

    def __init__(self, config: SLMConfig):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(config.d_emb, 4 * config.d_emb),
            GELU(),
            nn.Linear(4 * config.d_emb, config.d_emb)
        )

    def forward(self, x):
        return self.layers(x)


# Causal Multi-Head Attention  
class CausalMultiHeadAttention(nn.Module):
    """
    Multi-head causal self-attention
    """

    def __init__(self, config: SLMConfig):
        super().__init__()

        self.n_heads = config.n_heads
        self.head_dim = config.head_d_emb
        self.d_emb = config.d_emb

        # Q K V projections
        self.W_query = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)
        self.W_key = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)
        self.W_value = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)

        # output projection
        self.out_proj = nn.Linear(config.d_emb, config.d_emb)

    def forward(self, x):

        batch, tokens, d_emb = x.shape

        # create Q K V
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        return Q, K, V


# Causal Self-Attention Block
class CausalMultiHeadAttention(nn.Module):
    """
    Multi-head causal self-attention with masking
    """

    def __init__(self, config: SLMConfig):
        super().__init__()

        self.n_heads = config.n_heads
        self.head_dim = config.head_d_emb
        self.d_emb = config.d_emb

        # Q K V projections
        self.W_query = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)
        self.W_key = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)
        self.W_value = nn.Linear(config.d_emb, config.d_emb, bias=config.qkv_bias)

        # Output projection
        self.out_proj = nn.Linear(config.d_emb, config.d_emb)

        # Dropout
        self.attn_dropout = nn.Dropout(config.drop_rate)

        # Causal mask
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(config.n_blocks, config.n_blocks), diagonal=1)
        )

    def forward(self, x):

        batch, tokens, _ = x.shape

        # Q K V projections
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        # Split into heads
        Q = Q.view(batch, tokens, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch, tokens, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch, tokens, self.n_heads, self.head_dim).transpose(1, 2)

        # Attention scores
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        # Apply causal mask
        mask = self.mask[:tokens, :tokens].bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Softmax
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        # Context vector
        context = attn_weights @ V

        # Merge heads
        context = context.transpose(1, 2).contiguous().view(batch, tokens, self.d_emb)

        # Final projection
        output = self.out_proj(context)

        return output
    
# Transformer Block
class TransformerBlock(nn.Module):
    """Single transformer decoder block"""

    def __init__(self, config: SLMConfig):
        super().__init__()

        self.norm1 = LayerNorm(config)
        self.attention = CausalMultiHeadAttention(config)

        self.norm2 = LayerNorm(config)
        self.feed_forward = FeedForward(config)

    def forward(self, x):

        # Attention + residual
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = x + shortcut

        # FeedForward + residual
        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = x + shortcut

        return x


class GPT(nn.Module):
    """Complete GPT decoder model."""
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embedding
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_emb)
        # Position embedding (using n_blocks from your config)
        self.position_embedding = nn.Embedding(config.n_blocks, config.d_emb)

        # Transformer blocks
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config.n_layers)]
        )
        # FIXED: Passing config object, not just the integer
        self.final_norm = LayerNorm(config)

        # Output head
        self.output_head = nn.Linear(config.d_emb, config.vocab_size, bias=False)

        # Weight tying
        self.token_embedding.weight = self.output_head.weight

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.shape

        # Check sequence length against n_blocks
        assert t <= self.config.n_blocks, f"Sequence length {t} exceeds context {self.config.n_blocks}"

        tok_emb = self.token_embedding(idx)
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        pos_emb = self.position_embedding(pos)
        x = tok_emb + pos_emb
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.n_blocks else idx[:, -self.config.n_blocks:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = torch.softmax(logits, dim=-1)

            if top_p is not None:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                sorted_probs[cumulative_probs > top_p] = 0
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                next_token = torch.multinomial(sorted_probs, 1)
                idx_next = sorted_indices.gather(-1, next_token)
            else:
                idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


In [4]:
import torch
from datasets import load_dataset
import tiktoken

config = SLMConfig()
model = GPT(config)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model parameters:", sum(p.numel() for p in model.parameters()))

dataset = load_dataset("roneneldan/TinyStories", split="train")


Model parameters: 51017216


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [10]:
tokenizer = tiktoken.get_encoding("gpt2")

dataset = dataset.select(range(20000))

tokens = []

for example in dataset:
    tokens.extend(tokenizer.encode(example["text"]))

tokens = torch.tensor(tokens, dtype=torch.long)

In [11]:
block_size = config.n_blocks
batch_size = 16


def get_batch():

    ix = torch.randint(len(tokens) - block_size, (batch_size,))
    
    x = torch.stack([tokens[i:i+block_size] for i in ix])
    y = torch.stack([tokens[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(10000):

    x, y = get_batch()

    logits, loss = model(x, y)

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    if step % 100 == 0:
        print("Step:", step, "Loss:", loss.item())

Step: 0 Loss: 11.00888442993164
Step: 100 Loss: 6.05316162109375
Step: 200 Loss: 5.949949264526367
Step: 300 Loss: 5.925370693206787
Step: 400 Loss: 5.891244888305664
Step: 500 Loss: 5.644809246063232
Step: 600 Loss: 5.338775634765625
Step: 700 Loss: 5.166784286499023
Step: 800 Loss: 5.064808368682861
Step: 900 Loss: 5.09114408493042
Step: 1000 Loss: 4.784156799316406
Step: 1100 Loss: 4.847128391265869
Step: 1200 Loss: 4.807914733886719
Step: 1300 Loss: 4.548728942871094
Step: 1400 Loss: 4.622468948364258
Step: 1500 Loss: 4.552423477172852
Step: 1600 Loss: 4.706982612609863
Step: 1700 Loss: 4.455692291259766
Step: 1800 Loss: 4.40004825592041
Step: 1900 Loss: 4.400491714477539
Step: 2000 Loss: 4.5780792236328125
Step: 2100 Loss: 4.405864238739014
Step: 2200 Loss: 4.263355255126953
Step: 2300 Loss: 4.229891777038574
Step: 2400 Loss: 4.252256393432617
Step: 2500 Loss: 4.277590751647949
Step: 2600 Loss: 4.238708019256592
Step: 2700 Loss: 4.2790045738220215
Step: 2800 Loss: 4.00312805175781

In [12]:
print("Total tokens:", len(tokens))

Total tokens: 4446061


In [13]:
torch.save(model.state_dict(), "/kaggle/working/best_model_parameter.pt")

In [14]:
model.eval()
test_prompt = "Once upon a time"
context = torch.tensor([tokenizer.encode(test_prompt)]).to(device)
generated = model.generate(context, max_new_tokens=50, temperature=0.8)
print("Generated:", tokenizer.decode(generated[0].tolist()))


Generated: Once upon a time there was a little girl. She was very curious and loved exploring. Every day she would record her brush the new world car.

One day Daisy went to the park to find a pile of buildings to take. She was really excited. She
